# Домашнее задание 1

## 1. Выбор модели и контекста применения

**Область:** Computer Vision

**Семейство моделей:** GAN

**Задача:** Реализовать мобильный инференс легковесной модели для трансформации фотографий в стиль аниме

Доступ к модели планируется осуществлять через мобильное приложение. Пользователь делает фото или загружает его из галереи, нажимает на кнопку для обработки и затем получает результат. В данном случае предполагается небольшая частота запросов к модели ввиду локальной работы приложения. Требования ко времени ответа - несколько секунд.

## 2. Целевая аппаратная платформа

**Устройство:** Xiaomi Redmi Note 9 Pro

**Процессор:** Qualcomm Snapdragon 720G

**Ядра CPU:** 2 ядра Kryo 465 Gold (Cortex-A76) на 2300 МГц и 6 ядер Kryo 465 Silver (Cortex-A55) на 1800 МГц

**GPU:** Adreno 618, поддерживает API Vulkan 1.1, OpenGL ES 3.2 и DirectX 12

**APU (AI Processing Unit):** Hexagon 692 DSP

**Разрядность:** 64 бит

**Частота:** 2300 МГц

**Оперативная память:** тип LPDDR4X, частота 1866 МГц, объём 6 ГБ

**Энергопотребление:** 5 Вт

## 3. Архитектура модели

Для анализа выбрана модель `AnimeGAN` - первая версия в серии моделей `AnimeGAN`. Цель модели - придавать входным изображениям аниме-стиль. 

Как и любой GAN, модель состоит из двух блоков: генератор и дискриминатор. Ниже показаны архитектуры генератора и дискриминатора модели:

![Архитектуры генератора и дискриминатора](pictures/v1_architecture.png)

**Строение генератора:**
- представляет собой симметричный энкодер-декодер
- состоит из стандартных сверток, поканальных сепарабельных сверток (depthwise separable convolutions), инвертированных residual-блоков
(inverted residual blocks (IRBs)), а также upsampling и downsampling слоев
- в конце генератора происходит свертка с ядром 1x1 и затем используется функция активации Tanh

**Строение структурных блоков генератора:**

- ConvBlock состоит из стандартной свертки с ядром 3x3, instance-нормализации и функции активации LeakyReLU
- DSConv состоит из поканальной свертки с ядром 3x3, instance-нормализации, активации LeakyReLU и ConvBlock с ядром 1x1
- IRB содержит ConvBlock с ядром 1x1, поканальную свертку, поточечные свертки, слои instance-нормализации и активацию LeakyReLU
- в качестве downsampling и upsampling модулей используются Down-Conv и Up-Conv соответственно, в строении которых присутствуют блоки DSConv

![Строение структурных блоков генератора](pictures/v1_blocks.png)

**Строение дискриминатора:**
- состоит из последовательности стандартных сверток с ядром 3x3, активаций LeakyReLU и instance-нормализаций. В конце происходит заключительная свертка с 1 каналом, и по выходу сети определяется, реальное изображение или сгенерированное.

**Архитектурные особенности:**
- использование в середине генератора 8 последовательных идентичных инвертированных residual-блоков вместо стандартных residual-блоков. Это позволило  значительно уменьшить количество параметров и вычислительную сложность генератора

**Предположительные наиболее ресурсоемкие части модели:**
1. Первые и последние слои сети - по времени (из-за работы с большим разрешением)
2. Глубокие сверточные слои с большим количеством каналов - по времени выполнения и объему вычислений

## 4. Вычислительные затраты и узкие места

На данном этапе выполним инференс и профилирование модели на персональном компьютере (CPU Intel Core i7-1165G, 32Гб RAM) с использованием PyTorch.

In [1]:
import torch
from torch.profiler import profile, ProfilerActivity, record_function

from inference import auto_load_weight

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = auto_load_weight('weights/generator_hayao.pth', map_location=device)
num_params = sum(p.numel() for p in model.parameters())
print(f'Количество параметров модели: {num_params:_}')

Количество параметров модели: 3_757_184


In [3]:
named_layers = list(dict(model.named_modules()).keys())
print('Layers:')
for l in named_layers:
    print(l)

Layers:

encode_blocks
encode_blocks.0
encode_blocks.0.pad
encode_blocks.0.conv
encode_blocks.0.ins_norm
encode_blocks.0.activation
encode_blocks.1
encode_blocks.1.pad
encode_blocks.1.conv
encode_blocks.1.ins_norm
encode_blocks.1.activation
encode_blocks.2
encode_blocks.2.conv1
encode_blocks.2.conv1.depthwise
encode_blocks.2.conv1.pointwise
encode_blocks.2.conv1.ins_norm1
encode_blocks.2.conv1.activation1
encode_blocks.2.conv1.ins_norm2
encode_blocks.2.conv1.activation2
encode_blocks.2.conv2
encode_blocks.2.conv2.depthwise
encode_blocks.2.conv2.pointwise
encode_blocks.2.conv2.ins_norm1
encode_blocks.2.conv2.activation1
encode_blocks.2.conv2.ins_norm2
encode_blocks.2.conv2.activation2
encode_blocks.3
encode_blocks.3.pad
encode_blocks.3.conv
encode_blocks.3.ins_norm
encode_blocks.3.activation
encode_blocks.4
encode_blocks.4.depthwise
encode_blocks.4.pointwise
encode_blocks.4.ins_norm1
encode_blocks.4.activation1
encode_blocks.4.ins_norm2
encode_blocks.4.activation2
encode_blocks.5
encode

In [4]:
input = torch.randn(1, 3, 512, 512).to(device)

# warm-up
model(input)

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities += [ProfilerActivity.CUDA]

with profile(activities=activities, profile_memory=True, record_shapes=True, acc_events=True) as prof:
    with record_function("model_inference"):
        output = model(input)

sort_key = device.type + "_time_total"

print(prof.key_averages().table(sort_by=sort_key))

prof.export_chrome_trace("animegan_v1_trace.json")

c:\Users\angelina.kholicheva\AppData\Local\anaconda3\envs\ml_env\Lib\site-packages\torch\nn\modules\instancenorm.py:115: UserWarning: input's size at dim=1 does not match num_features. You can silence this warning by not passing in num_features, which is not used because affine=False
  warnings.warn(


--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                 model_inference         0.46%      16.281ms       100.00%        3.571s        3.571s       3.96 Gb    -131.00 Mb             1  
                    aten::conv2d         0.02%     587.300us        68.77%        2.455s      50.112ms       1.68 Gb           0 b            49  
               aten::convolution         0.05%       1.806ms        68.75%        2.455s      50.098ms       1.68 Gb           0 b            49  
              aten::_convolution         0.04%       1.359ms        68.70%        2.453s      50.062ms       1.68 Gb  

Видим, что наибольшая часть времени - 68.57% тратится на операции свертки (в частности, на `aten::mkldnn_convolution` для PyTorch). Также 19.48% времени уходит на операцию `aten::native_batch_norm`. Наибольшие затраты памяти приходятся на операцию `aten::empty`, которая используется для создания тензора заданного размера без инициализации его значений.

## 5. Системные ограничения

### Ключевые ограничения системы

**Аппаратная платформа (целевая):**

**Процессор:** Qualcomm Snapdragon 720G (8nm, 8 ядер Kryo 465 до 2.3 ГГц)

**DSP/AI акселератор:** Hexagon 692 DSP (оптимизирован для INT8 вычислений)

**GPU:** Adreno 618

**ОЗУ**: 6 ГБ

**Энергопотребление:** 5 Вт (активное охлаждение отсутствует)

**Требования к инференсу (для сценария "мобильный фильтр в реальном времени"):**

**1. Задержка (Latency):**

*Желательно:* < 200 мс на кадр (чтобы обработка не ощущалась как "подтормаживание")

*Критично:* < 500 мс (иначе пользователь закроет приложение)

*Анализ профилирования:* Текущий инференс на CPU занимает ~3.6 секунды (событие model_inference, dur: 3570555.7 мкс). Это в 18 раз больше желаемой цели. Перенос на ARM CPU будет ещё медленнее.

**2. Пропускная способность (Throughput):**

Для одиночных фото достаточно 1 кадра в 1-2 секунды.

**3. Память (Memory):**

*ОЗУ (RAM):* Модель должна укладываться в < 200-300 МБ в пике, чтобы не вытеснять другие приложения и не вызывать Out-Of-Memory (OOM).

*Постоянная память (ROM):* Размер файла модели < 20-30 МБ (чтобы не раздувать размер APK).

*Анализ профилирования:* Пиковое потребление памяти на CPU составляет ~4.3 ГБ, что катастрофически много для мобильного устройства. Основная причина — хранение промежуточных тензоров в формате FP32.

**4. Энергопотребление:**

*Критично:* Нагрев корпуса и разряд батареи. Непрерывная работа не должна приводить к троттлингу за 5-10 минут. INT8 вычисления на DSP потребляют значительно меньше энергии, чем FP32 на CPU.

### Критичные ограничения для выбранного сценария

Для сценария "мобильный фильтр для фото" самым критичным ограничением является задержка в совокупности с потреблением памяти. Если модель не сможет обработать кадр за разумное время (менее 500 мс) или вызовет OOM, она будет бесполезна. Энергопотребление вторично, но важно для пользовательского опыта.

## 6. Гипотезы по ускорению

На основе анализа профилирования (где основные затраты пришлись на операции `aten::convolution`, `aten::mkldnn_convolution` и `aten::native_batch_norm`) и целевой платформы, предлагаю следующие методы:

### 1. Квантование: преобразование весов и активаций из FP32 в INT8

**Затрагиваемые части:** все слои, содержащие операции `aten::convolution` и `aten::native_batch_norm`.

**Механизм ускорения:**
- уменьшение размера модели в 4 раза (с ~15 МБ до ~3.75 МБ)
- снижение потребления памяти для промежуточных тензоров в 4 раза
- сспользование аппаратных INT8 инструкций на CPU/GPU, и самое главное — возможность запуска на Hexagon DSP, который работает с INT8 в десятки раз эффективнее, чем CPU с FP32

### 2. Прунинг: удаление наименее важных весов или целых фильтров из сверточных слоев

**Затрагиваемые части:** слои с большим количеством каналов, например, переходы 64 → 128 → 256 → 512. В bottleneck присутствует 512 каналов, и не все одинаково важны.

**Механизм ускорения:**
- сокращение числа операций и, соответственно, времени инференса
- дополнительное уменьшение размера модели

### 3. Слияние операций (Kernel / Operator Fusion): объединение последовательных операций в одну

**Затрагиваемые части:** практически каждый блок `aten::convolution`, сразу после которого идет `aten::batch_norm` и `aten::leaky_relu`.

**Механизм ускорения:**
- сокращение количества проходов по памяти
- уменьшение накладных расходов на вызов отдельных ядер

### 4. Упрощение архитектуры + дистилляция знаний: замена тяжелых блоков на более легкие аналоги

**Затрагиваемые части:** все сверточные слои.

**Механизм ускорения:** 
- depthwise convolution требует в разы меньше вычислений, чем обычная, при сопоставимом качестве
- использование более легкой сети-ученика

### 5. Снижение разрешения входного изображения: подавать на вход изображение меньшего разрешения.

**Затрагиваемые части:** все слои энкодера и декодера, количество вычислений падает квадратично.

**Механизм ускорения:** уменьшение разрешения в 2 раза уменьшает количество операций примерно в 4 раза.

### Вывод:

В качестве основного метода ускорения и оптимизации модели будет использовано квантование. Затем, в зависимости от результата, планируется попробовать дистилляцию знаний.

## 7. Инженерные компромиссы

Опишем, какой выигрыш дает каждый описанный выше метод и чем приходится платить за его использование.

| Метод ускорения | Выигрыш | Плата |
| --------------- | ------- | ----- |
| Квантование в INT8 | Ускорение в ~10 раз, уменьшение размера модели, снижение потребления памяти в ~4 раза | Небольшая потеря качества |
| Прунинг / Дистилляция | Дополнительное ускорение в 1.5-2 раза | Время на дообучение, риск ухудшения качества |
| Снижение разрешения входа | Ускорение | Потеря мелких деталей изображения |
| Запуск на DSP | Максимальная скорость, низкое энергопотребление | Зависимость от платформы (Qualcomm) |

## Источники

1. Оригинальный репозиторий: https://github.com/TachibanaYoshino/AnimeGAN
2. Версия на PyTorch: https://github.com/ptran1203/pytorch-animeGAN